# Data Preprocessing for MarketPredictor

This notebook prepares the data for model training: cleaning, feature selection, temporal split, normalization, and saving.

## 1. Import libraries and load enriched data
Load the datasets resulting from feature engineering.

In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load enriched datasets
sp500 = pd.read_csv('../data/processed/sp500_features.csv', parse_dates=['Date'])
cac40 = pd.read_csv('../data/processed/cac40_features.csv', parse_dates=['Date'])

## 2. Data cleaning
Remove rows with NaN (generated by rolling windows) and handle outliers if needed.

In [ ]:
# Smart cleaning: impute exogenous features (ETF), only drop if target or critical columns are NaN
# Critical columns to never impute
critical_cols = ['Target', 'Close', 'Open', 'High', 'Low', 'Volume']
exog_cols_sp500 = [col for col in sp500.columns if col.startswith('sector_')]
exog_cols_cac40 = [col for col in cac40.columns if col.startswith('sector_')]

# Impute exogenous features (ETF) with ffill
sp500[exog_cols_sp500] = sp500[exog_cols_sp500].ffill()
cac40[exog_cols_cac40] = cac40[exog_cols_cac40].ffill()

# Only drop rows where critical columns are NaN
sp500_clean = sp500.dropna(subset=critical_cols).reset_index(drop=True)
cac40_clean = cac40.dropna(subset=critical_cols).reset_index(drop=True)

print(f'SP500: {len(sp500)} rows before, {len(sp500_clean)} rows after cleaning.')
print(f'CAC40: {len(cac40)} rows before, {len(cac40_clean)} rows after cleaning.')

SP500 : 2812 lignes avant, 2812 lignes après nettoyage.
CAC40 : 2863 lignes avant, 2863 lignes après nettoyage.


## 3. Selection of relevant features
Choose the columns to use for modeling (features + target).

In [11]:
features_sp500 = [col for col in sp500_clean.columns if col not in ['Date', 'Target', 'Close', 'Open', 'High', 'Low', 'Volume'] and not col.endswith('_future')]
features_cac40 = [col for col in cac40_clean.columns if col not in ['Date', 'Target', 'Close', 'Open', 'High', 'Low', 'Volume'] and not col.endswith('_future')]

X_sp500 = sp500_clean[features_sp500]
y_sp500 = sp500_clean['Target']

X_cac40 = cac40_clean[features_cac40]
y_cac40 = cac40_clean['Target']

## 4. Temporal train/test split
We split the data while respecting chronological order to avoid any data leakage.

In [12]:
# Split 80% train, 20% test
split_idx_sp500 = int(0.8 * len(X_sp500))
X_train_sp500, X_test_sp500 = X_sp500.iloc[:split_idx_sp500], X_sp500.iloc[split_idx_sp500:]
y_train_sp500, y_test_sp500 = y_sp500.iloc[:split_idx_sp500], y_sp500.iloc[split_idx_sp500:]

split_idx_cac40 = int(0.8 * len(X_cac40))
X_train_cac40, X_test_cac40 = X_cac40.iloc[:split_idx_cac40], X_cac40.iloc[split_idx_cac40:]
y_train_cac40, y_test_cac40 = y_cac40.iloc[:split_idx_cac40], y_cac40.iloc[split_idx_cac40:]

## 5. Feature normalization/standardization
We standardize the features to facilitate training for certain models (optional for RandomForest, recommended for others).

In [13]:
scaler_sp500 = StandardScaler()
X_train_sp500_scaled = scaler_sp500.fit_transform(X_train_sp500)
X_test_sp500_scaled = scaler_sp500.transform(X_test_sp500)

scaler_cac40 = StandardScaler()
X_train_cac40_scaled = scaler_cac40.fit_transform(X_train_cac40)
X_test_cac40_scaled = scaler_cac40.transform(X_test_cac40)

## 6. Saving datasets ready for modeling
We save the numpy arrays or DataFrames for training and testing.

In [14]:
# Sauvegarde au format numpy (rapide pour le ML)
np.save('../data/processed/X_train_sp500.npy', X_train_sp500_scaled)
np.save('../data/processed/X_test_sp500.npy', X_test_sp500_scaled)
np.save('../data/processed/y_train_sp500.npy', y_train_sp500)
np.save('../data/processed/y_test_sp500.npy', y_test_sp500)

np.save('../data/processed/X_train_cac40.npy', X_train_cac40_scaled)
np.save('../data/processed/X_test_cac40.npy', X_test_cac40_scaled)
np.save('../data/processed/y_train_cac40.npy', y_train_cac40)
np.save('../data/processed/y_test_cac40.npy', y_test_cac40)

In [ ]:
# Save the list of features used for training (exact order)
with open('../models/cac40_features_used.txt', 'w') as f:
    for col in features_cac40:
        f.write(f"{col}\n")
with open('../models/sp500_features_used.txt', 'w') as f:
    for col in features_sp500:
        f.write(f"{col}\n")